# PS S6E6 - 024 RealMLP seed ensemble and blend check

Experiment: `exp_20260608_024_seed_ensemble_and_blend_check`

Purpose:
- Evaluate RealMLP seed ensemble candidates using `015/021/022/023`.
- Check whether RealMLP seed variants add value to current top candidates `019/020`.
- Re-check blend interaction with `014` and `018`.
- Compute OOF correlation, disagreement, error overlap, and safe blends.
- Save only the best safe blend OOF/pred `.npy` files for later bias search.

Current context:
- 019: Public-confirmed current best, CV `0.968872315017554`, Public LB `0.97000`
- 020: CV-trusted attack, CV `0.9692240443617589`, Public LB `0.96969`
- 015: stable RealMLP as-is backup, CV `0.9681693449222352`, Public LB `0.96977`
- 021/022/023: RealMLP seed variants to test as ensemble/blend material.

Main question:
- Do RealMLP seed variants improve the current 019/020 branch?
- If yes, next is 025 bias search on 024 best.
- If no, stop seed-variant expansion and keep 019/020/017/015 as main candidates.

In [1]:

import os
import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from scipy.optimize import minimize
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.linear_model import Ridge, ElasticNet

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260608_024_seed_ensemble_and_blend_check"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

MODEL_SPECS = [
    {"key":"014_blend_bias","exp_id":"exp_20260605_014_bias_search_on_013_best_blend","family":"Blend","role":"old_blend_bias_reference","oof":"oof_exp_20260605_014_bias_search_on_013_best_blend_proba.npy","pred":"pred_exp_20260605_014_bias_search_on_013_best_blend_proba.npy","cv":0.9660085788215015,"public_lb":0.96703},
    {"key":"015_realmlp_seed42","exp_id":"exp_20260605_015_shared001_realmlp_as_is","family":"RealMLP","role":"stable_single_realmlp_backup","oof":"oof_exp_20260605_015_shared001_realmlp_as_is_proba.npy","pred":"pred_exp_20260605_015_shared001_realmlp_as_is_proba.npy","cv":0.9681693449222352,"public_lb":0.96977},
    {"key":"017_realmlp_bias","exp_id":"exp_20260606_017_bias_search_on_015_realmlp","family":"RealMLP","role":"previous_best_realmlp_bias_backup","oof":"oof_exp_20260606_017_bias_search_on_015_realmlp_proba.npy","pred":"pred_exp_20260606_017_bias_search_on_015_realmlp_proba.npy","cv":0.9683022653955233,"public_lb":0.96985},
    {"key":"018_xgb_shared003","exp_id":"exp_20260606_018_shared003_xgb_as_is","family":"XGBoost","role":"effective_blend_material","oof":"oof_exp_20260606_018_shared003_xgb_as_is_proba.npy","pred":"pred_exp_20260606_018_shared003_xgb_as_is_proba.npy","cv":0.965207986418628,"public_lb":0.96578},
    {"key":"019_blend_best","exp_id":"exp_20260607_019_blend_add018_xgb_check","family":"Blend","role":"public_confirmed_current_best","oof":"oof_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy","pred":"pred_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy","cv":0.968872315017554,"public_lb":0.97000},
    {"key":"020_blend_bias","exp_id":"exp_20260607_020_bias_search_on_019_best_blend","family":"Blend","role":"cv_trusted_attack","oof":"oof_exp_20260607_020_bias_search_on_019_best_blend_proba.npy","pred":"pred_exp_20260607_020_bias_search_on_019_best_blend_proba.npy","cv":0.9692240443617589,"public_lb":0.96969},
    {"key":"021_realmlp_seed777","exp_id":"exp_20260607_021_shared001_realmlp_seed777","family":"RealMLP","role":"seed_variant_mixed_seed777","oof":"oof_exp_20260607_021_shared001_realmlp_seed777_proba.npy","pred":"pred_exp_20260607_021_shared001_realmlp_seed777_proba.npy","cv":0.9682532829014837,"public_lb":0.96926},
    {"key":"022_realmlp_seed2026","exp_id":"exp_20260608_022_shared001_realmlp_seed2026","family":"RealMLP","role":"seed_variant_clean_seed2026","oof":"oof_exp_20260608_022_shared001_realmlp_seed2026_proba.npy","pred":"pred_exp_20260608_022_shared001_realmlp_seed2026_proba.npy","cv":0.9681106610656959,"public_lb":0.96956},
    {"key":"023_realmlp_seed3407","exp_id":"exp_20260608_023_shared001_realmlp_seed3407","family":"RealMLP","role":"seed_variant_clean_seed3407","oof":"oof_exp_20260608_023_shared001_realmlp_seed3407_proba.npy","pred":"pred_exp_20260608_023_shared001_realmlp_seed3407_proba.npy","cv":0.9680674611199329,"public_lb":0.96925},
]

BLEND_SETS = {
    "A_019_only": ["019_blend_best"],
    "B_020_only": ["020_blend_bias"],
    "C_017_only": ["017_realmlp_bias"],
    "D_015_only": ["015_realmlp_seed42"],

    # Pure seed ensembles.
    "E_seed_015_021": ["015_realmlp_seed42", "021_realmlp_seed777"],
    "F_seed_015_022": ["015_realmlp_seed42", "022_realmlp_seed2026"],
    "G_seed_015_023": ["015_realmlp_seed42", "023_realmlp_seed3407"],
    "H_seed_015_021_022": ["015_realmlp_seed42", "021_realmlp_seed777", "022_realmlp_seed2026"],
    "I_seed_015_021_022_023": ["015_realmlp_seed42", "021_realmlp_seed777", "022_realmlp_seed2026", "023_realmlp_seed3407"],

    # Bias RealMLP plus seeds.
    "J_017_021_022_023": ["017_realmlp_bias", "021_realmlp_seed777", "022_realmlp_seed2026", "023_realmlp_seed3407"],
    "K_015_017_021_022_023": ["015_realmlp_seed42", "017_realmlp_bias", "021_realmlp_seed777", "022_realmlp_seed2026", "023_realmlp_seed3407"],

    # Add seeds to current 019/020 branches.
    "L_019_021_022_023": ["019_blend_best", "021_realmlp_seed777", "022_realmlp_seed2026", "023_realmlp_seed3407"],
    "M_020_021_022_023": ["020_blend_bias", "021_realmlp_seed777", "022_realmlp_seed2026", "023_realmlp_seed3407"],
    "N_019_020_021_022_023": ["019_blend_best", "020_blend_bias", "021_realmlp_seed777", "022_realmlp_seed2026", "023_realmlp_seed3407"],

    # Rebuild top blend with seed candidates.
    "O_014_017_018_021_022_023": ["014_blend_bias", "017_realmlp_bias", "018_xgb_shared003", "021_realmlp_seed777", "022_realmlp_seed2026", "023_realmlp_seed3407"],
    "P_014_018_019_021_022_023": ["014_blend_bias", "018_xgb_shared003", "019_blend_best", "021_realmlp_seed777", "022_realmlp_seed2026", "023_realmlp_seed3407"],
    "Q_014_018_020_021_022_023": ["014_blend_bias", "018_xgb_shared003", "020_blend_bias", "021_realmlp_seed777", "022_realmlp_seed2026", "023_realmlp_seed3407"],
    "R_014_017_018_019_020_021_022_023": ["014_blend_bias", "017_realmlp_bias", "018_xgb_shared003", "019_blend_best", "020_blend_bias", "021_realmlp_seed777", "022_realmlp_seed2026", "023_realmlp_seed3407"],
}

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)
print("NPY_DIR:", NPY_DIR)


EXP_ID: exp_20260608_024_seed_ensemble_and_blend_check
OUTDIR: /kaggle/working/exp_20260608_024_seed_ensemble_and_blend_check
NPY_DIR: /kaggle/input/datasets/mizushimatoshihiko/npy-files


In [2]:
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def load_npy_from_dataset(filename):
    path = NPY_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing npy file: {path}\n"
            f"Add the output file to the npy-files Kaggle dataset or edit MODEL_SPECS."
        )
    return path

def validate_proba(name, arr, n_rows, n_classes):
    assert arr.shape == (n_rows, n_classes), (name, arr.shape, n_rows, n_classes)
    assert np.isfinite(arr).all(), name
    row_sum = arr.sum(axis=1)
    assert np.allclose(row_sum, 1.0, atol=1e-4), (name, row_sum.min(), row_sum.max())
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

def normalize_rows(p, eps=1e-12):
    p = np.clip(p, eps, None)
    return p / p.sum(axis=1, keepdims=True)

def softmax_np(x, axis=1):
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def proba_to_pred(p):
    return np.argmax(p, axis=1)

def flatten_proba(p):
    return np.asarray(p, dtype=float).reshape(-1)

def corr_pearson(a, b):
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def corr_spearman(a, b):
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(spearmanr(a, b).correlation)

def class_recalls(y_true, y_pred, class_names):
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out

def evaluate_proba(y_true, p, class_names):
    pred = proba_to_pred(p)
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    res.update(class_recalls(y_true, pred, class_names))
    return res

def rank_normalize_matrix(p):
    out = np.zeros_like(p, dtype=np.float64)
    n = p.shape[0]
    for j in range(p.shape[1]):
        r = pd.Series(p[:, j]).rank(method="average").to_numpy()
        out[:, j] = (r - 1) / max(n - 1, 1)
    return normalize_rows(out)

def logit_transform(p, eps=1e-15):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def build_meta_features(keys, proba_dict, mode):
    mats = []
    for k in keys:
        p = proba_dict[k]
        if mode == "raw":
            mats.append(p)
        elif mode == "rank":
            mats.append(rank_normalize_matrix(p))
        elif mode == "logit":
            mats.append(logit_transform(p))
        elif mode == "raw_logit":
            mats.append(p)
            mats.append(logit_transform(p))
        elif mode == "raw_rank_logit":
            mats.append(p)
            mats.append(rank_normalize_matrix(p))
            mats.append(logit_transform(p))
        else:
            raise ValueError(mode)
    return np.concatenate(mats, axis=1)

def average_proba(keys, oof_dict, pred_dict=None):
    oof_avg = normalize_rows(np.mean([oof_dict[k] for k in keys], axis=0))
    pred_avg = None
    if pred_dict is not None:
        pred_avg = normalize_rows(np.mean([pred_dict[k] for k in keys], axis=0))
    return oof_avg, pred_avg

def hill_climb_weights(y_true, probas, steps=(0.1, 0.05, 0.02, 0.01, 0.005, 0.002), max_iter=300):
    n = len(probas)
    w = np.ones(n, dtype=float) / n
    cur = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    cur_score = balanced_accuracy_score(y_true, cur.argmax(axis=1))
    hist = [{"iter": 0, "score": float(cur_score), "weights": w.copy().tolist()}]

    for step in steps:
        improved = True
        it = 0
        while improved and it < max_iter:
            improved = False
            it += 1
            best_score = cur_score
            best_w = w.copy()
            for i in range(n):
                for direction in [-1, 1]:
                    cand_w = w.copy()
                    cand_w[i] += direction * step
                    cand_w = np.clip(cand_w, 0.0, None)
                    if cand_w.sum() <= 0:
                        continue
                    cand_w = cand_w / cand_w.sum()
                    cand = normalize_rows(sum(cand_w[j] * probas[j] for j in range(n)))
                    score = balanced_accuracy_score(y_true, cand.argmax(axis=1))
                    if score > best_score + 1e-15:
                        best_score = score
                        best_w = cand_w.copy()
            if best_score > cur_score + 1e-15:
                cur_score = best_score
                w = best_w
                improved = True
                hist.append({"iter": len(hist), "score": float(cur_score), "weights": w.copy().tolist()})

    final = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    return w, final, float(cur_score), hist

def nm_optimize_weights(y_true, probas, maxiter=1500):
    n = len(probas)
    def unpack(theta):
        e = np.exp(theta - np.max(theta))
        return e / e.sum()
    def objective(theta):
        w = unpack(theta)
        p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
        return -balanced_accuracy_score(y_true, p.argmax(axis=1))
    res = minimize(objective, np.zeros(n), method="Nelder-Mead",
                   options={"maxiter": maxiter, "xatol": 1e-7, "fatol": 1e-10})
    w = unpack(res.x)
    p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    score = balanced_accuracy_score(y_true, p.argmax(axis=1))
    return w, p, float(score), res

def onehot_y(y, n_classes):
    out = np.zeros((len(y), n_classes), dtype=np.float64)
    out[np.arange(len(y)), y] = 1.0
    return out

In [3]:
for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)
if not NPY_DIR.exists():
    raise FileNotFoundError(NPY_DIR)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)
assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof = {}
pred = {}
resolved_specs = []

for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = load_npy_from_dataset(spec["oof"])
    pred_path = load_npy_from_dataset(spec["pred"])

    oof_arr = np.load(oof_path).astype(np.float32)
    pred_arr = np.load(pred_path).astype(np.float32)

    validate_proba(f"{key} oof", oof_arr, len(train), n_classes)
    validate_proba(f"{key} pred", pred_arr, len(test), n_classes)

    oof[key] = oof_arr
    pred[key] = pred_arr

    row = dict(spec)
    row["resolved_oof_path"] = str(oof_path)
    row["resolved_pred_path"] = str(pred_path)
    row["oof_shape"] = list(oof_arr.shape)
    row["pred_shape"] = list(pred_arr.shape)
    resolved_specs.append(row)

model_keys = [s["key"] for s in MODEL_SPECS]

print("train:", train.shape)
print("test :", test.shape)
print("class_names:", class_names)
display(pd.DataFrame(resolved_specs))

train: (577347, 12)
test : (247435, 11)
class_names: ['GALAXY', 'QSO', 'STAR']


,key,exp_id,family,role,oof,pred,cv,public_lb,resolved_oof_path,resolved_pred_path,oof_shape,pred_shape
0,014_blend_bias,exp_20260605_014_bias_search_on_013_best_blend,Blend,old_blend_bias_reference,oof_exp_20260605_014_bias_search_on_013_best_b...,pred_exp_20260605_014_bias_search_on_013_best_...,0.966009,0.96703,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
1,015_realmlp_seed42,exp_20260605_015_shared001_realmlp_as_is,RealMLP,stable_single_realmlp_backup,oof_exp_20260605_015_shared001_realmlp_as_is_p...,pred_exp_20260605_015_shared001_realmlp_as_is_...,0.968169,0.96977,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
2,017_realmlp_bias,exp_20260606_017_bias_search_on_015_realmlp,RealMLP,previous_best_realmlp_bias_backup,oof_exp_20260606_017_bias_search_on_015_realml...,pred_exp_20260606_017_bias_search_on_015_realm...,0.968302,0.96985,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
3,018_xgb_shared003,exp_20260606_018_shared003_xgb_as_is,XGBoost,effective_blend_material,oof_exp_20260606_018_shared003_xgb_as_is_proba...,pred_exp_20260606_018_shared003_xgb_as_is_prob...,0.965208,0.96578,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
4,019_blend_best,exp_20260607_019_blend_add018_xgb_check,Blend,public_confirmed_current_best,oof_exp_20260607_019_blend_add018_xgb_check_be...,pred_exp_20260607_019_blend_add018_xgb_check_b...,0.968872,0.97000,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
5,020_blend_bias,exp_20260607_020_bias_search_on_019_best_blend,Blend,cv_trusted_attack,oof_exp_20260607_020_bias_search_on_019_best_b...,pred_exp_20260607_020_bias_search_on_019_best_...,0.969224,0.96969,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
6,021_realmlp_seed777,exp_20260607_021_shared001_realmlp_seed777,RealMLP,seed_variant_mixed_seed777,oof_exp_20260607_021_shared001_realmlp_seed777...,pred_exp_20260607_021_shared001_realmlp_seed77...,0.968253,0.96926,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
7,022_realmlp_seed2026,exp_20260608_022_shared001_realmlp_seed2026,RealMLP,seed_variant_clean_seed2026,oof_exp_20260608_022_shared001_realmlp_seed202...,pred_exp_20260608_022_shared001_realmlp_seed20...,0.968111,0.96956,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
8,023_realmlp_seed3407,exp_20260608_023_shared001_realmlp_seed3407,RealMLP,seed_variant_clean_seed3407,oof_exp_20260608_023_shared001_realmlp_seed340...,pred_exp_20260608_023_shared001_realmlp_seed34...,0.968067,0.96925,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"


In [4]:
rows = []
for spec in resolved_specs:
    key = spec["key"]
    p = oof[key]
    y_pred = proba_to_pred(p)
    score = balanced_accuracy_score(y, y_pred)

    row = {
        "key": key,
        "exp_id": spec["exp_id"],
        "family": spec["family"],
        "role": spec["role"],
        "cv_from_memo": spec.get("cv", np.nan),
        "public_lb": spec.get("public_lb", np.nan),
        "recomputed_oof_cv": float(score),
        "delta_recomputed_minus_memo": float(score - spec.get("cv", score)),
    }
    row.update(class_recalls(y, y_pred, class_names))
    rows.append(row)

individual_df = pd.DataFrame(rows).sort_values("recomputed_oof_cv", ascending=False)
display(individual_df)
individual_df.to_csv(OUTDIR / "individual_scores.csv", index=False)

,key,exp_id,family,role,cv_from_memo,public_lb,recomputed_oof_cv,delta_recomputed_minus_memo,recall_GALAXY,recall_QSO,recall_STAR
5,020_blend_bias,exp_20260607_020_bias_search_on_019_best_blend,Blend,cv_trusted_attack,0.969224,0.96969,0.969224,0.0,0.961137,0.976200,0.970335
4,019_blend_best,exp_20260607_019_blend_add018_xgb_check,Blend,public_confirmed_current_best,0.968872,0.97000,0.968872,0.0,0.965479,0.974441,0.966696
2,017_realmlp_bias,exp_20260606_017_bias_search_on_015_realmlp,RealMLP,previous_best_realmlp_bias_backup,0.968302,0.96985,0.968302,0.0,0.959889,0.975867,0.969150
6,021_realmlp_seed777,exp_20260607_021_shared001_realmlp_seed777,RealMLP,seed_variant_mixed_seed777,0.968253,0.96926,0.968253,0.0,0.960022,0.975987,0.968752
1,015_realmlp_seed42,exp_20260605_015_shared001_realmlp_as_is,RealMLP,stable_single_realmlp_backup,0.968169,0.96977,0.968169,0.0,0.962234,0.976098,0.966177
7,022_realmlp_seed2026,exp_20260608_022_shared001_realmlp_seed2026,RealMLP,seed_variant_clean_seed2026,0.968111,0.96956,0.968111,0.0,0.959113,0.976226,0.968993
8,023_realmlp_seed3407,exp_20260608_023_shared001_realmlp_seed3407,RealMLP,seed_variant_clean_seed3407,0.968067,0.96925,0.968067,0.0,0.959036,0.975859,0.969308
0,014_blend_bias,exp_20260605_014_bias_search_on_013_best_blend,Blend,old_blend_bias_reference,0.966009,0.96703,0.966009,0.0,0.968062,0.970617,0.959347
3,018_xgb_shared003,exp_20260606_018_shared003_xgb_as_is,XGBoost,effective_blend_material,0.965208,0.96578,0.965208,0.0,0.966870,0.972043,0.956711


In [5]:
pair_rows = []
pred_labels = {k: proba_to_pred(oof[k]) for k in model_keys}
wrong_flags = {k: pred_labels[k] != y for k in model_keys}

for a, b in combinations(model_keys, 2):
    pa, pb = oof[a], oof[b]
    ya, yb = pred_labels[a], pred_labels[b]
    wrong_a, wrong_b = wrong_flags[a], wrong_flags[b]
    both = wrong_a & wrong_b
    either = wrong_a | wrong_b

    row = {
        "model_a": a,
        "model_b": b,
        "pearson_flat_proba": corr_pearson(flatten_proba(pa), flatten_proba(pb)),
        "spearman_flat_proba": corr_spearman(flatten_proba(pa), flatten_proba(pb)),
        "argmax_agreement": float((ya == yb).mean()),
        "argmax_disagreement": float((ya != yb).mean()),
        "wrong_rate_a": float(wrong_a.mean()),
        "wrong_rate_b": float(wrong_b.mean()),
        "both_wrong_rate": float(both.mean()),
        "either_wrong_rate": float(either.mean()),
        "error_jaccard": float(both.sum() / max(either.sum(), 1)),
        "a_wrong_b_correct_rate": float((wrong_a & ~wrong_b).mean()),
        "a_correct_b_wrong_rate": float((~wrong_a & wrong_b).mean()),
    }
    for ci, cls in enumerate(class_names):
        row[f"pearson_proba_{cls}"] = corr_pearson(pa[:, ci], pb[:, ci])
        row[f"spearman_proba_{cls}"] = corr_spearman(pa[:, ci], pb[:, ci])
    pair_rows.append(row)

pairwise_df = pd.DataFrame(pair_rows).sort_values("pearson_flat_proba")
display(pairwise_df)
pairwise_df.to_csv(OUTDIR / "pairwise_oof_correlation.csv", index=False)

pearson_flat = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
spearman_flat = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
argmax_agreement = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
error_jaccard = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)

for a in model_keys:
    for b in model_keys:
        pearson_flat.loc[a, b] = corr_pearson(flatten_proba(oof[a]), flatten_proba(oof[b]))
        spearman_flat.loc[a, b] = corr_spearman(flatten_proba(oof[a]), flatten_proba(oof[b]))
        argmax_agreement.loc[a, b] = float((pred_labels[a] == pred_labels[b]).mean())
        both = wrong_flags[a] & wrong_flags[b]
        either = wrong_flags[a] | wrong_flags[b]
        error_jaccard.loc[a, b] = float(both.sum() / max(either.sum(), 1))

display(pearson_flat)
display(spearman_flat)
display(argmax_agreement)
display(error_jaccard)

pearson_flat.to_csv(OUTDIR / "corr_matrix_pearson_flat_proba.csv")
spearman_flat.to_csv(OUTDIR / "corr_matrix_spearman_flat_proba.csv")
argmax_agreement.to_csv(OUTDIR / "matrix_argmax_agreement.csv")
error_jaccard.to_csv(OUTDIR / "matrix_error_jaccard.csv")

,model_a,model_b,pearson_flat_proba,spearman_flat_proba,argmax_agreement,argmax_disagreement,wrong_rate_a,wrong_rate_b,both_wrong_rate,either_wrong_rate,error_jaccard,a_wrong_b_correct_rate,a_correct_b_wrong_rate,pearson_proba_GALAXY,spearman_proba_GALAXY,pearson_proba_QSO,spearman_proba_QSO,pearson_proba_STAR,spearman_proba_STAR
24,018_xgb_shared003,022_realmlp_seed2026,0.990274,0.907060,0.981153,0.018847,0.033536,0.035999,0.025480,0.044055,0.578376,0.008056,0.010519,0.987040,0.924417,0.993964,0.820350,0.980955,0.771333
25,018_xgb_shared003,023_realmlp_seed3407,0.990468,0.898508,0.981159,0.018841,0.033536,0.036079,0.025543,0.044072,0.579564,0.007993,0.010536,0.987391,0.924045,0.994134,0.814852,0.981101,0.746140
23,018_xgb_shared003,021_realmlp_seed777,0.990511,0.910208,0.981519,0.018481,0.033536,0.035488,0.025420,0.043605,0.582959,0.008116,0.010068,0.987380,0.927871,0.994193,0.832084,0.981127,0.766964
15,017_realmlp_bias,018_xgb_shared003,0.990670,0.908650,0.981524,0.018476,0.035542,0.033536,0.025439,0.043639,0.582933,0.010103,0.008097,0.987587,0.939083,0.994756,0.857918,0.980882,0.780807
9,015_realmlp_seed42,018_xgb_shared003,0.991146,0.918772,0.982243,0.017757,0.034388,0.033536,0.025227,0.042697,0.590848,0.009161,0.008309,0.988216,0.939921,0.994776,0.854090,0.981740,0.778653
5,014_blend_bias,021_realmlp_seed777,0.993098,0.911363,0.984462,0.015538,0.032668,0.035488,0.026466,0.041691,0.634815,0.006203,0.009022,0.991128,0.934720,0.995234,0.835716,0.987046,0.775531
6,014_blend_bias,022_realmlp_seed2026,0.993168,0.907927,0.984434,0.015566,0.032668,0.035999,0.026698,0.041970,0.636127,0.005970,0.009301,0.991205,0.930758,0.995251,0.819241,0.987453,0.776736
7,014_blend_bias,023_realmlp_seed3407,0.993367,0.898459,0.984644,0.015356,0.032668,0.036079,0.026857,0.041890,0.641141,0.005811,0.009221,0.991492,0.929608,0.995451,0.805961,0.987692,0.754841
1,014_blend_bias,017_realmlp_bias,0.993651,0.904324,0.985206,0.014794,0.032668,0.035542,0.026857,0.041353,0.649466,0.005811,0.008685,0.991905,0.941093,0.995901,0.851459,0.987737,0.773884
2,014_blend_bias,018_xgb_shared003,0.993720,0.976021,0.984379,0.015621,0.032668,0.033536,0.025427,0.040778,0.623540,0.007242,0.008110,0.991644,0.967257,0.996035,0.894090,0.986881,0.925413


,014_blend_bias,015_realmlp_seed42,017_realmlp_bias,018_xgb_shared003,019_blend_best,020_blend_bias,021_realmlp_seed777,022_realmlp_seed2026,023_realmlp_seed3407
014_blend_bias,1.000000,0.994167,0.993651,0.993720,0.997309,0.996506,0.993098,0.993168,0.993367
015_realmlp_seed42,0.994167,1.000000,0.999895,0.991146,0.998530,0.998570,0.997029,0.996858,0.996965
017_realmlp_bias,0.993651,0.999895,1.000000,0.990670,0.998337,0.998634,0.997043,0.996929,0.997066
018_xgb_shared003,0.993720,0.991146,0.990670,1.000000,0.996517,0.995772,0.990511,0.990274,0.990468
019_blend_best,0.997309,0.998530,0.998337,0.996517,1.000000,0.999766,0.996696,0.996579,0.996746
020_blend_bias,0.996506,0.998570,0.998634,0.995772,0.999766,1.000000,0.996898,0.996895,0.997085
021_realmlp_seed777,0.993098,0.997029,0.997043,0.990511,0.996696,0.996898,1.000000,0.996806,0.996952
022_realmlp_seed2026,0.993168,0.996858,0.996929,0.990274,0.996579,0.996895,0.996806,1.000000,0.996983
023_realmlp_seed3407,0.993367,0.996965,0.997066,0.990468,0.996746,0.997085,0.996952,0.996983,1.000000


,014_blend_bias,015_realmlp_seed42,017_realmlp_bias,018_xgb_shared003,019_blend_best,020_blend_bias,021_realmlp_seed777,022_realmlp_seed2026,023_realmlp_seed3407
014_blend_bias,1.000000,0.916308,0.904324,0.976021,0.936711,0.930534,0.911363,0.907927,0.898459
015_realmlp_seed42,0.916308,1.000000,0.997811,0.918772,0.992035,0.989531,0.899250,0.891457,0.888005
017_realmlp_bias,0.904324,0.997811,1.000000,0.908650,0.990462,0.990901,0.896449,0.887723,0.886267
018_xgb_shared003,0.976021,0.918772,0.908650,1.000000,0.937942,0.933916,0.910208,0.907060,0.898508
019_blend_best,0.936711,0.992035,0.990462,0.937942,1.000000,0.997020,0.906204,0.898200,0.895837
020_blend_bias,0.930534,0.989531,0.990901,0.933916,0.997020,1.000000,0.907776,0.898587,0.898472
021_realmlp_seed777,0.911363,0.899250,0.896449,0.910208,0.906204,0.907776,1.000000,0.892382,0.888887
022_realmlp_seed2026,0.907927,0.891457,0.887723,0.907060,0.898200,0.898587,0.892382,1.000000,0.881866
023_realmlp_seed3407,0.898459,0.888005,0.886267,0.898508,0.895837,0.898472,0.888887,0.881866,1.000000


,014_blend_bias,015_realmlp_seed42,017_realmlp_bias,018_xgb_shared003,019_blend_best,020_blend_bias,021_realmlp_seed777,022_realmlp_seed2026,023_realmlp_seed3407
014_blend_bias,1.000000,0.985937,0.985206,0.984379,0.990153,0.988416,0.984462,0.984434,0.984644
015_realmlp_seed42,0.985937,1.000000,0.997825,0.982243,0.993035,0.993354,0.989838,0.989672,0.989873
017_realmlp_bias,0.985206,0.997825,1.000000,0.981524,0.992258,0.993202,0.989764,0.989625,0.989925
018_xgb_shared003,0.984379,0.982243,0.981524,1.000000,0.988840,0.987474,0.981519,0.981153,0.981159
019_blend_best,0.990153,0.993035,0.992258,0.988840,1.000000,0.996248,0.989443,0.989041,0.989325
020_blend_bias,0.988416,0.993354,0.993202,0.987474,0.996248,1.000000,0.989663,0.989516,0.989873
021_realmlp_seed777,0.984462,0.989838,0.989764,0.981519,0.989443,0.989663,1.000000,0.989512,0.989725
022_realmlp_seed2026,0.984434,0.989672,0.989625,0.981153,0.989041,0.989516,0.989512,1.000000,0.989660
023_realmlp_seed3407,0.984644,0.989873,0.989925,0.981159,0.989325,0.989873,0.989725,0.989660,1.000000


,014_blend_bias,015_realmlp_seed42,017_realmlp_bias,018_xgb_shared003,019_blend_best,020_blend_bias,021_realmlp_seed777,022_realmlp_seed2026,023_realmlp_seed3407
014_blend_bias,1.000000,0.659451,0.649466,0.623540,0.742478,0.710850,0.634815,0.636127,0.641141
015_realmlp_seed42,0.659451,1.000000,0.941244,0.590848,0.814570,0.827230,0.750694,0.748548,0.752714
017_realmlp_bias,0.649466,0.941244,1.000000,0.582933,0.798627,0.825950,0.752521,0.750986,0.757032
018_xgb_shared003,0.623540,0.590848,0.582933,1.000000,0.714710,0.692699,0.582959,0.578376,0.579564
019_blend_best,0.742478,0.814570,0.798627,0.714710,1.000000,0.894809,0.735646,0.727836,0.734694
020_blend_bias,0.710850,0.827230,0.825950,0.692699,0.894809,1.000000,0.747221,0.744663,0.753023
021_realmlp_seed777,0.634815,0.750694,0.752521,0.582959,0.735646,0.747221,1.000000,0.748263,0.753108
022_realmlp_seed2026,0.636127,0.748548,0.750986,0.578376,0.727836,0.744663,0.748263,1.000000,0.752906
023_realmlp_seed3407,0.641141,0.752714,0.757032,0.579564,0.734694,0.753023,0.753108,0.752906,1.000000


In [6]:
blend_rows = {}

def record_blend(set_name, method, keys, oof_p, test_p=None, weights=None, extra=None):
    ev = evaluate_proba(y, oof_p, class_names)
    row = {
        "set_name": set_name,
        "method": method,
        "keys": ",".join(keys),
        "n_models": len(keys),
        "balanced_accuracy": ev["balanced_accuracy"],
        "weights_json": json.dumps({k: float(w) for k, w in zip(keys, weights)}, ensure_ascii=False) if weights is not None else None,
    }
    row.update({k: v for k, v in ev.items() if k != "balanced_accuracy"})
    if extra:
        row.update(extra)
    blend_rows[(set_name, method)] = row

for set_name, keys in BLEND_SETS.items():
    print(f"===== {set_name}: {keys} =====")
    probas = [oof[k] for k in keys]
    pred_probas = [pred[k] for k in keys]

    avg_oof, avg_pred = average_proba(keys, oof, pred)
    record_blend(set_name, "avg", keys, avg_oof, avg_pred, weights=np.ones(len(keys)) / len(keys))

    if len(keys) >= 2:
        hc_w, hc_oof, hc_score, hc_hist = hill_climb_weights(y, probas)
        hc_pred = normalize_rows(sum(hc_w[i] * pred_probas[i] for i in range(len(keys))))
        record_blend(set_name, "hc_nonnegative_raw", keys, hc_oof, hc_pred, weights=hc_w, extra={"hc_history_len": len(hc_hist)})

        nm_w, nm_oof, nm_score, nm_res = nm_optimize_weights(y, probas)
        nm_pred = normalize_rows(sum(nm_w[i] * pred_probas[i] for i in range(len(keys))))
        record_blend(set_name, "nm_softmax_raw", keys, nm_oof, nm_pred, weights=nm_w, extra={"nm_success": bool(nm_res.success), "nm_message": str(nm_res.message)})

    # Diagnostic only: in-sample meta screening, not safe final OOF.
    for mode in ["raw", "logit", "raw_logit", "raw_rank_logit"]:
        X_meta = build_meta_features(keys, oof, mode)
        X_test_meta = build_meta_features(keys, pred, mode)
        try:
            lr = LogisticRegression(C=0.05, penalty="l2", solver="lbfgs", max_iter=2000, random_state=SEED, n_jobs=-1)
            lr.fit(X_meta, y)
            lr_oof = lr.predict_proba(X_meta)
            lr_pred = lr.predict_proba(X_test_meta)
            record_blend(set_name, f"logreg_{mode}", keys, lr_oof, lr_pred, extra={"C": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] logreg failed: {set_name} {mode}: {e}")
        try:
            rc = RidgeClassifier(alpha=10.0, random_state=SEED)
            rc.fit(X_meta, y)
            rc_oof = softmax_np(rc.decision_function(X_meta), axis=1)
            rc_pred = softmax_np(rc.decision_function(X_test_meta), axis=1)
            record_blend(set_name, f"ridgeclf_{mode}", keys, rc_oof, rc_pred, extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridgeclf failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            rr = Ridge(alpha=10.0, random_state=SEED)
            rr.fit(X_meta, y_oh)
            rr_oof = normalize_rows(np.clip(rr.predict(X_meta), 1e-12, None))
            rr_pred = normalize_rows(np.clip(rr.predict(X_test_meta), 1e-12, None))
            record_blend(set_name, f"ridge_reg_{mode}", keys, rr_oof, rr_pred, extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridge_reg failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            en_oof_list, en_test_list = [], []
            for c in range(n_classes):
                en = ElasticNet(alpha=0.0005, l1_ratio=0.05, random_state=SEED, max_iter=2000)
                en.fit(X_meta, y_oh[:, c])
                en_oof_list.append(en.predict(X_meta))
                en_test_list.append(en.predict(X_test_meta))
            en_oof = normalize_rows(np.clip(np.vstack(en_oof_list).T, 1e-12, None))
            en_pred = normalize_rows(np.clip(np.vstack(en_test_list).T, 1e-12, None))
            record_blend(set_name, f"elasticnet_reg_{mode}", keys, en_oof, en_pred, extra={"alpha": 0.0005, "l1_ratio": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] elasticnet_reg failed: {set_name} {mode}: {e}")

blend_df = pd.DataFrame(list(blend_rows.values())).sort_values("balanced_accuracy", ascending=False)
display(blend_df.head(150))
blend_df.to_csv(OUTDIR / "blend_diagnostics.csv", index=False)

===== A_019_only: ['019_blend_best'] =====
===== B_020_only: ['020_blend_bias'] =====
===== C_017_only: ['017_realmlp_bias'] =====
[WARN] logreg failed: C_017_only logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: C_017_only logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] ridge_reg failed: C_017_only logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] elasticnet_reg failed: C_017_only logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] logreg failed: C_017_only raw_logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: C_017_only raw_logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] ridge_reg failed: C_017_only raw_logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] elasticnet_reg failed: C_017_only raw_logit: Input X contains infinit

,set_name,method,keys,n_models,balanced_accuracy,weights_json,recall_GALAXY,recall_QSO,recall_STAR,C,risk,alpha,l1_ratio,hc_history_len,nm_success,nm_message
108,N_019_020_021_022_023,hc_nonnegative_raw,"019_blend_best,020_blend_bias,021_realmlp_seed...",5,0.969480,"{""019_blend_best"": 0.1720919461492222, ""020_bl...",0.961956,0.976174,0.970311,NaN,NaN,NaN,NaN,13.0,NaN,NaN
129,Q_014_018_020_021_022_023,hc_nonnegative_raw,"014_blend_bias,018_xgb_shared003,020_blend_bia...",6,0.969466,"{""014_blend_bias"": 0.055071905766532786, ""018_...",0.962761,0.975918,0.969719,NaN,NaN,NaN,NaN,12.0,NaN,NaN
101,M_020_021_022_023,hc_nonnegative_raw,"020_blend_bias,021_realmlp_seed777,022_realmlp...",4,0.969464,"{""020_blend_bias"": 0.5578658812432313, ""021_re...",0.961248,0.976396,0.970746,NaN,NaN,NaN,NaN,13.0,NaN,NaN
136,R_014_017_018_019_020_021_022_023,hc_nonnegative_raw,"014_blend_bias,017_realmlp_bias,018_xgb_shared...",8,0.969412,"{""014_blend_bias"": 0.1342293970927078, ""017_re...",0.963307,0.975611,0.969320,NaN,NaN,NaN,NaN,13.0,NaN,NaN
94,L_019_021_022_023,hc_nonnegative_raw,"019_blend_best,021_realmlp_seed777,022_realmlp...",4,0.969350,"{""019_blend_best"": 0.548194684586193, ""021_rea...",0.963407,0.975577,0.969066,NaN,NaN,NaN,NaN,13.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6,A_019_only,ridgeclf_logit,019_blend_best,1,0.959464,None,0.978934,0.962584,0.936874,NaN,in_sample_meta_screening,10.0000,NaN,NaN,NaN,NaN
7,A_019_only,ridge_reg_logit,019_blend_best,1,0.959464,None,0.978934,0.962584,0.936874,NaN,in_sample_meta_screening,10.0000,NaN,NaN,NaN,NaN
25,B_020_only,elasticnet_reg_logit,020_blend_bias,1,0.959309,None,0.978960,0.962345,0.936621,NaN,in_sample_meta_screening,0.0005,0.05,NaN,NaN,NaN
23,B_020_only,ridgeclf_logit,020_blend_bias,1,0.959300,None,0.978982,0.962311,0.936608,NaN,in_sample_meta_screening,10.0000,NaN,NaN,NaN,NaN


In [7]:
safe_methods = ["avg", "hc_nonnegative_raw", "nm_softmax_raw"]
safe_blend_df = blend_df[blend_df["method"].isin(safe_methods)].copy()
safe_blend_df = safe_blend_df.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)

best_row = safe_blend_df.iloc[0].to_dict()
best_set_name = best_row["set_name"]
best_method = best_row["method"]
best_keys = best_row["keys"].split(",")

print("Best safe blend:")
print(json.dumps(best_row, indent=2, ensure_ascii=False))

if best_method == "avg":
    best_oof_proba, best_pred_proba = average_proba(best_keys, oof, pred)
elif best_method in ["hc_nonnegative_raw", "nm_softmax_raw"]:
    probas = [oof[k] for k in best_keys]
    pred_probas = [pred[k] for k in best_keys]
    weights_json = json.loads(best_row["weights_json"])
    w = np.array([weights_json[k] for k in best_keys], dtype=float)
    best_oof_proba = normalize_rows(sum(w[i] * probas[i] for i in range(len(best_keys))))
    best_pred_proba = normalize_rows(sum(w[i] * pred_probas[i] for i in range(len(best_keys))))
else:
    raise RuntimeError(best_method)

validate_proba("best_oof_proba", best_oof_proba, len(train), n_classes)
validate_proba("best_pred_proba", best_pred_proba, len(test), n_classes)

best_oof_score = balanced_accuracy_score(y, best_oof_proba.argmax(axis=1))
print("best_oof_score:", best_oof_score)

best_oof_npy = OUTDIR / "oof_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy"
best_pred_npy = OUTDIR / "pred_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy"

np.save(best_oof_npy, best_oof_proba.astype(np.float32))
np.save(best_pred_npy, best_pred_proba.astype(np.float32))

best_labels = le.inverse_transform(best_pred_proba.argmax(axis=1))
submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET_COL: best_labels,
})
assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

best_submission_path = OUTDIR / "submission_exp_20260608_024_seed_ensemble_and_blend_check_best_blend.csv"
submission.to_csv(best_submission_path, index=False)

best_info = {
    "exp_id": EXP_ID,
    "best_set_name": best_set_name,
    "best_method": best_method,
    "best_keys": best_keys,
    "cv_score": float(best_oof_score),
    "weights_json": best_row.get("weights_json"),
    "oof_path": str(best_oof_npy),
    "pred_path": str(best_pred_npy),
    "submission_path": str(best_submission_path),
    "row": best_row,
    "note": "Best safe blend excludes in-sample meta screening methods.",
}
save_json(best_info, OUTDIR / "best_blend_info.json")

saved_submission_df = pd.DataFrame([{
    "set_name": best_set_name,
    "method": best_method,
    "balanced_accuracy": float(best_oof_score),
    "submission": best_submission_path.name,
    "oof_proba": best_oof_npy.name,
    "pred_proba": best_pred_npy.name,
}])
display(saved_submission_df)
saved_submission_df.to_csv(OUTDIR / "saved_submission_candidates.csv", index=False)

Best safe blend:
{
  "set_name": "N_019_020_021_022_023",
  "method": "hc_nonnegative_raw",
  "keys": "019_blend_best,020_blend_bias,021_realmlp_seed777,022_realmlp_seed2026,023_realmlp_seed3407",
  "n_models": 5,
  "balanced_accuracy": 0.9694803109983217,
  "weights_json": "{\"019_blend_best\": 0.1720919461492222, \"020_blend_bias\": 0.4147680653120452, \"021_realmlp_seed777\": 0.1520334427906098, \"022_realmlp_seed2026\": 0.12458465124038007, \"023_realmlp_seed3407\": 0.1365218945077428}",
  "recall_GALAXY": 0.9619556002967045,
  "recall_QSO": 0.9761744192994887,
  "recall_STAR": 0.9703109133987718,
  "C": NaN,
  "risk": NaN,
  "alpha": NaN,
  "l1_ratio": NaN,
  "hc_history_len": 13.0,
  "nm_success": NaN,
  "nm_message": NaN
}
best_oof_score: 0.9694803109983217


,set_name,method,balanced_accuracy,submission,oof_proba,pred_proba
0,N_019_020_021_022_023,hc_nonnegative_raw,0.96948,submission_exp_20260608_024_seed_ensemble_and_...,oof_exp_20260608_024_seed_ensemble_and_blend_c...,pred_exp_20260608_024_seed_ensemble_and_blend_...


In [8]:
# ============================================================
# 8. Role judgement
# ============================================================

def score_of(key):
    return float(individual_df.loc[individual_df["key"] == key, "recomputed_oof_cv"].iloc[0])

judgement = {
    "reference_scores": {
        "019_cv": 0.968872315017554,
        "019_public_lb": 0.97000,
        "020_cv": 0.9692240443617589,
        "020_public_lb": 0.96969,
        "017_cv": 0.9683022653955233,
        "017_public_lb": 0.96985,
        "015_cv": 0.9681693449222352,
        "015_public_lb": 0.96977,
        "021_cv": 0.9682532829014837,
        "021_public_lb": 0.96926,
        "022_cv": 0.9681106610656959,
        "022_public_lb": 0.96956,
        "023_cv": 0.9680674611199329,
        "023_public_lb": 0.96925,
    },
    "individual_best": {
        "key": individual_df.iloc[0]["key"],
        "cv": float(individual_df.iloc[0]["recomputed_oof_cv"]),
        "public_lb": float(individual_df.iloc[0]["public_lb"]),
    },
    "best_safe_blend": best_info,
    "main_questions": {
        "do_seed_variants_add_to_019": "Check L/P/R safe methods and seed weights.",
        "do_seed_variants_add_to_020": "Check M/Q/R safe methods and seed weights.",
        "is_pure_seed_ensemble_useful": "Check E/F/G/H/I seed ensemble rows.",
        "should_make_more_seed_variants": "Only if 021/022/023 retain meaningful nonzero weights.",
        "next_bias_target": "If 024 best safe blend improves over 019/020, run 025 bias search on 024 best.",
    },
    "initial_policy": {
        "019_blend_best": "public_confirmed_current_best_reference",
        "020_blend_bias": "cv_trusted_attack_reference",
        "017_realmlp_bias": "previous_best_backup",
        "015_realmlp_seed42": "stable_single_backup",
        "021_022_023": "seed_variant_material_under_test",
        "linear_meta_rows": "screening_only_not_final_without_nested_meta",
    },
}

save_json(judgement, OUTDIR / "role_judgement.json")
print(json.dumps(judgement, indent=2, ensure_ascii=False))

{
  "reference_scores": {
    "019_cv": 0.968872315017554,
    "019_public_lb": 0.97,
    "020_cv": 0.9692240443617589,
    "020_public_lb": 0.96969,
    "017_cv": 0.9683022653955233,
    "017_public_lb": 0.96985,
    "015_cv": 0.9681693449222352,
    "015_public_lb": 0.96977,
    "021_cv": 0.9682532829014837,
    "021_public_lb": 0.96926,
    "022_cv": 0.9681106610656959,
    "022_public_lb": 0.96956,
    "023_cv": 0.9680674611199329,
    "023_public_lb": 0.96925
  },
  "individual_best": {
    "key": "020_blend_bias",
    "cv": 0.9692240443617589,
    "public_lb": 0.96969
  },
  "best_safe_blend": {
    "exp_id": "exp_20260608_024_seed_ensemble_and_blend_check",
    "best_set_name": "N_019_020_021_022_023",
    "best_method": "hc_nonnegative_raw",
    "best_keys": [
      "019_blend_best",
      "020_blend_bias",
      "021_realmlp_seed777",
      "022_realmlp_seed2026",
      "023_realmlp_seed3407"
    ],
    "cv_score": 0.9694803109983217,
    "weights_json": "{\"019_blend_best\": 

In [9]:
# ============================================================
# 9. Save summary / memo
# ============================================================

summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "status": "completed",
    "purpose": "RealMLP seed ensemble and blend check with 015/021/022/023 plus 014/018/019/020 references",
    "npy_dir": str(NPY_DIR),
    "model_keys": model_keys,
    "class_names": class_names,
    "resolved_specs": resolved_specs,
    "individual_scores": individual_df.to_dict(orient="records"),
    "pairwise_oof_correlation": pairwise_df.to_dict(orient="records"),
    "blend_diagnostics": blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics": safe_blend_df.to_dict(orient="records"),
    "best_blend_info": best_info,
    "saved_submission_candidates": saved_submission_df.to_dict(orient="records"),
    "role_judgement": judgement,
}
save_json(summary, OUTDIR / "seed_ensemble_and_blend_summary.json")

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "RealMLP seed ensemble and blend check",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "created_at": "2026-06-08",
    },
    "objective": {
        "summary": (
            "Load OOF/pred probabilities from the fixed npy-files Kaggle dataset folder, "
            "evaluate RealMLP seed ensemble candidates 015/021/022/023, "
            "and test whether those seed variants add value to 019/020/014/018 blend branches. "
            "Save only the best safe blend OOF/pred proba for the next experiment."
        ),
        "success_criteria": [
            "load 014/015/017/018/019/020/021/022/023 OOF and pred npy files",
            "recompute individual balanced accuracy",
            "compute pairwise OOF correlations",
            "evaluate pure RealMLP seed ensembles",
            "evaluate seed variants added to 019/020 branches",
            "separate safe blends from in-sample meta screening",
            "save only best safe blend OOF/pred npy",
            "save best safe blend submission",
            "save memo and summary",
        ],
    },
    "inputs": {
        "npy_dir": str(NPY_DIR),
        "models": resolved_specs,
        "blend_sets": BLEND_SETS,
    },
    "outputs": {
        "individual_scores": "individual_scores.csv",
        "pairwise_oof_correlation": "pairwise_oof_correlation.csv",
        "corr_matrix_pearson_flat_proba": "corr_matrix_pearson_flat_proba.csv",
        "corr_matrix_spearman_flat_proba": "corr_matrix_spearman_flat_proba.csv",
        "matrix_argmax_agreement": "matrix_argmax_agreement.csv",
        "matrix_error_jaccard": "matrix_error_jaccard.csv",
        "blend_diagnostics": "blend_diagnostics.csv",
        "best_blend_info": "best_blend_info.json",
        "saved_submission_candidates": "saved_submission_candidates.csv",
        "best_oof_proba": best_oof_npy.name,
        "best_pred_proba": best_pred_npy.name,
        "best_submission": best_submission_path.name,
        "role_judgement": "role_judgement.json",
        "summary": "seed_ensemble_and_blend_summary.json",
    },
    "judgement": {
        "status": "completed_pending_manual_review",
        "initial_view": (
            "021/022/023 are weaker as single models than 015/017/019/020. "
            "Their value depends entirely on whether they add complementary signal in safe blends. "
            "In-sample meta rows are diagnostic only."
        ),
        "next_action": [
            "Review individual_scores.csv",
            "Review pairwise_oof_correlation.csv for seed diversity",
            "Review pure seed ensemble rows E/F/G/H/I",
            "Review whether 021/022/023 receive nonzero weights in best safe blends",
            "Submit best safe blend only if it improves over 019/020 enough",
            "If best safe blend improves, run 025 bias search on 024 best",
            "If seed variants get zero/near-zero weight, stop making more seed variants",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)

Saved files:
 - best_blend_info.json
 - blend_diagnostics.csv
 - corr_matrix_pearson_flat_proba.csv
 - corr_matrix_spearman_flat_proba.csv
 - individual_scores.csv
 - matrix_argmax_agreement.csv
 - matrix_error_jaccard.csv
 - memo.yml
 - oof_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy
 - pairwise_oof_correlation.csv
 - pred_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy
 - role_judgement.json
 - saved_submission_candidates.csv
 - seed_ensemble_and_blend_summary.json
 - submission_exp_20260608_024_seed_ensemble_and_blend_check_best_blend.csv
